In [ ]:
import os 
os.chdir("/hpc/home/ephdh/workspace/suzhou_false_validation/data")

import glob
import pydicom
import numpy as np 
import pandas as pd
from tqdm import tqdm
from datetime import datetime
from tableone import TableOne
from collections import Counter
from pydicom.misc import is_dicom

# Re-organize meta data

In [ ]:
tn_excel = pd.read_excel('meta_data/真阴性名单.xlsx', sheet_name='True Negatives')
tn_excel['Group'] = 'TN'
tn_excel = tn_excel.rename(columns={'Patient Name': 'PatientName', 
                                    'Patient Age': 'PatientAge',
                                    'Density category': 'DensityCategory',
                                    'Lesion type': 'LesionType', 
                                    'BI-RADS risk': 'BIRADSRisk', 
                                    'Histological subtype': 'HistologicalSubtype'
                                    }
                                    )

In [ ]:
tn_excel.head()

In [ ]:
tp_excel = pd.read_excel('meta_data/真阳性名单钼靶.xlsx', sheet_name='True Positives')
tp_excel = tp_excel.rename(columns={'PatientName': 'PatientName', 
                                    'PatientAge': 'PatientAge',
                                    'Density category': 'DensityCategory',
                                    ' Lesion type': 'LesionType', 
                                    ' BI-RADS risk': 'BIRADSRisk', 
                                    'Histological subtypes': 'HistologicalSubtype'
                                    }
                                    )
include_cols = ['PatientName', 'PatientAge', 'DensityCategory', 'LesionType', 'BIRADSRisk', 
                'HistologicalSubtype']
tp_excel = tp_excel[include_cols]
tp_excel['Group'] = 'TP'

In [ ]:
# tp_excel.columns

In [ ]:
tp_excel.head()

In [ ]:
fn_excel = pd.read_excel('meta_data/假阴性名单.xls', sheet_name='original_dcm_info')
fn_excel = fn_excel.rename(columns={'PatientName': 'PatientName', 
                                    'PatientAge': 'PatientAge',
                                    'Density category': 'DensityCategory',
                                    ' Lesion type': 'LesionType', 
                                    ' BI-RADS risk': 'BIRADSRisk', 
                                    'Histological subtypes': 'HistologicalSubtype'
                                    }
                                    )
fn_excel = fn_excel[include_cols]
fn_excel['Group'] = 'FN'

In [ ]:
# print(fn_excel.columns)
fn_excel.head()

In [ ]:
fp_excel = pd.read_excel('meta_data/假阳性名单.xlsx', sheet_name='Sheet1')
fp_excel = fp_excel.rename(columns={'Patient Name': 'PatientName', 
                                    'Patient Age': 'PatientAge',
                                    'Density category': 'DensityCategory',
                                    'Lesion type': 'LesionType', 
                                    'BI-RADS risk': 'BIRADSRisk', 
                                    'Histological subtype': 'HistologicalSubtype'
                                    }
                                    )
fp_excel = fp_excel[include_cols]
fp_excel['Group'] = 'FP'

In [ ]:
print(fp_excel.columns)
fp_excel.head()

In [ ]:
meta_data = pd.concat([tp_excel, tn_excel, fp_excel, fn_excel], ignore_index=True)
meta_data = meta_data[meta_data['HistologicalSubtype']!='请剔除'].reset_index(drop=True)

In [ ]:
meta_data.head()

In [ ]:
print(meta_data['Group'].value_counts())

In [ ]:
# print(meta_data['HistologicalSubtype'].value_counts())

In [ ]:
print(meta_data.shape, meta_data['PatientName'].nunique())

In [ ]:
duplidcated_meta = meta_data[meta_data.duplicated(subset=['PatientName'], 
                                                  keep=False)].sort_values(
                                                      by=['PatientName']).reset_index(drop=True)
print(duplidcated_meta.shape, duplidcated_meta['PatientName'].nunique())

In [ ]:
# print(non_duplidcated_meta['Group'].value_counts())

In [ ]:
duplidcated_meta

In [ ]:
meta_data.to_csv("/hpc/home/ephdh/workspace/suzhou_false_validation/data/meta_data/meta_data_raw.csv", 
                 index=False, encoding="utf-16")

#### Check result: Duplicated rows are just same names

## Standardize and categorize the columns

In [ ]:
meta_data_edited = pd.read_csv("/hpc/home/ephdh/workspace/suzhou_false_validation/data/meta_data/meta_data_raw_edited.csv",
                               encoding="utf-16", sep="\t")

In [ ]:
meta_data_edited.head()

In [ ]:
# Standardize Density

# Mapping to a canonical A/B/C/D code
density_map = {
    "a": "A", "A": "A",
    "b": "B", "B": "B",
    "c": "C", "C": "C",
    "d": "D", "D": "D",
    "中量腺体型": "B",   # adjust if you prefer C
    "混合型": "B",       # could also be C depending on your radiologist
    "a/b": "B",         # boundary -> choose B as intermediate
    "无": np.nan
}

def standardize_density(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return density_map.get(x, np.nan)

meta_data_edited["DensityCategory_std"] = meta_data_edited["DensityCategory"].apply(standardize_density)

# Optional: also map A/B/C/D -> 1/2/3/4 for modeling
density_numeric_map = {"A": 1, "B": 2, "C": 3, "D": 4}
meta_data_edited["DensityCategory_num"] = meta_data_edited["DensityCategory_std"].map(density_numeric_map)

# print(meta_data_edited[["DensityCategory", "DensityCategory_std", "DensityCategory_num"]].head(20))
print("DensityCategory_std value counts:")
print(meta_data_edited["DensityCategory_std"].value_counts(dropna=False))

In [ ]:
# Standardize BIRADS
def standardize_birads(x):
    """
    Return main BIRADS category as an integer (0-6), or np.nan.
    """
    if pd.isna(x):
        return np.nan
    x = str(x).strip().upper()

    # Handle 4A/4B/4C
    if x in ["4A", "4B", "4C"]:
        return 4

    # Plain integers '0'...'6'
    if x.isdigit():
        v = int(x)
        if 0 <= v <= 6:
            return v

    # Fallback
    return np.nan

def birads_subtype(x):
    """
    Extract subtype for BIRADS 4 (e.g., 4A/4B/4C), otherwise return None.
    """
    if pd.isna(x):
        return None
    x = str(x).strip().upper()
    if x in ["4A", "4B", "4C"]:
        return x
    return None

meta_data_edited["BIRADSRisk_std"] = meta_data_edited["BIRADSRisk"].apply(standardize_birads).astype("float")
meta_data_edited["BIRADSRisk_sub"] = meta_data_edited["BIRADSRisk"].apply(birads_subtype)

print("BIRADSRisk_std value counts:")
print(meta_data_edited["BIRADSRisk_std"].value_counts(dropna=False).sort_index())

In [ ]:
# Categorize LesionType

def categorize_lesion_type(x: str) -> str:
    if pd.isna(x):
        return "Unknown/NA"
    x = str(x).strip()

    # Explicit normal/benign
    if x in ["无异常", "无"] or "良性" in x or "增生" in x:
        return "Negative/Benign"

    # Simple placeholder or missing
    if x in ["-", "—"]:
        return "Unknown/NA"

    # Keywords (order matters: more specific patterns first)
    has_mass   = ("肿块" in x) or ("肿块影" in x)
    has_nodule = ("结节" in x) or ("结节影" in x) or ("结节样" in x)
    has_calc   = ("钙化" in x) or ("钙化灶" in x)
    has_arch   = ("结构紊乱" in x) or ("结构扭曲" in x) or ("结构稍乱" in x)
    has_asym   = ("不对称" in x) or ("非对称" in x) or ("局灶性不对称" in x) or ("局灶不对称" in x)
    has_dense  = ("致密影" in x) or ("高密度影" in x)

    # Mass / Nodule dominant
    if has_mass and not (has_calc or has_arch or has_asym or has_dense):
        return "Mass"
    if has_nodule and not (has_calc or has_arch or has_asym or has_dense):
        return "Nodule/FocalDensity"

    # Calcification dominant (no strong mass/nodule/arch/asym)
    if has_calc and not (has_mass or has_nodule or has_arch or has_asym):
        return "Calcification"

    # Architectural distortion / asymmetry dominant
    if (has_arch or has_asym or has_dense) and not (has_mass or has_nodule or has_calc):
        return "ArchitecturalDistortion/Asymmetry"

    # Mixed lesions
    if sum([has_mass, has_nodule, has_calc, has_arch, has_asym or has_dense]) >= 2:
        return "Mixed"

    # Fallbacks
    if has_mass:
        return "Mass"
    if has_nodule:
        return "Nodule/FocalDensity"
    if has_calc:
        return "Calcification"
    if has_arch or has_asym or has_dense:
        return "ArchitecturalDistortion/Asymmetry"

    return "Other"

# 2. Apply categorization
meta_data_edited["LesionType_cat"] = meta_data_edited["LesionType"].apply(categorize_lesion_type)

# 3. Inspect result
print("Category counts:")
print(meta_data_edited["LesionType_cat"].value_counts())

In [ ]:
def categorize_histology(x: str) -> str:
    if pd.isna(x):
        return "Unknown/NA"
    x = str(x).strip()

    # Malignant subtypes
    if "Paget" in x:
        return "Paget Disease"
    if any(k in x for k in ["浸润性导管癌", "非特殊型浸润性癌", "非特殊型浸润癌", "浸润性癌", "以导管原位癌为主的浸润性导管癌"]):
        return "Invasive Ductal Carcinoma"
    if "浸润性小叶癌" in x:
        return "Invasive Lobular Carcinoma"
    if "导管原位癌" in x:
        return "Ductal Carcinoma In Situ"
    if any(k in x for k in ["乳头状癌", "实性乳头状癌", "包裹性乳头状癌", "导管内乳头状病变"]):
        return "Papillary Carcinoma / Lesion"
    if any(k in x for k in ["黏液", "粘液", "混合型黏液", "黏液腺癌"]):
        return "Mucinous / Colloid Carcinoma"
    if any(k in x for k in ["大汗腺", "化生", "间叶分化"]):
        return "Apocrine / Metaplastic Carcinoma"
    if any(k in x for k in ["叶状肿瘤", "纤维上皮性肿瘤", "纤维腺瘤", "硬化性腺病", "管状腺瘤"]):
        return "Benign Fibroepithelial Lesion"
    if any(k in x for k in ["乳腺病", "导管扩张", "增生", "炎", "囊肿", "钙化"]):
        return "Benign Proliferative / Non-proliferative Disease"
    if any(k in x for k in ["无手术病理", "我院未手术"]):
        return "Unknown/NA"

    # Fallback
    return "Other"

# Apply to dataset
meta_data_edited["HistologicalSubtype_cat"] = meta_data_edited["HistologicalSubtype"].apply(categorize_histology)

# Show sample and counts
print("Category counts:")
print(meta_data_edited["HistologicalSubtype_cat"].value_counts())

In [ ]:
meta_data_edited.head()

### Coarser categorization

In [ ]:
df = meta_data_edited.copy()

# --- Density (coarse) ---

density_map = {
    "a": "A", "A": "A",
    "b": "B", "B": "B",
    "c": "C", "C": "C",
    "d": "D", "D": "D",
    "中量腺体型": "B",   # adjust to C if your radiologists prefer
    "混合型": "B",
    "a/b": "B",
    "无": np.nan
}

def standardize_density(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return density_map.get(x, np.nan)

df["Density_std"] = df["DensityCategory"].apply(standardize_density)

def coarse_density(x):
    if x in ["A", "B"]:
        return "Low density (A/B)"
    if x in ["C", "D"]:
        return "High density (C/D)"
    return "Unknown"

df["Density_coarse"] = df["Density_std"].apply(coarse_density)

In [ ]:
# --- BIRADS (coarse) ---

def birads_main(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    if s in ["4A", "4B", "4C"]:
        return 4
    if s.isdigit():
        v = int(s)
        if 0 <= v <= 6:
            return v
    return np.nan

def birads_coarse(x):
    if pd.isna(x):
        return "Unknown"
    s = str(x).strip().upper()
    # preserve original string for subtype checks
    if s == "0":
        return "0 Incomplete"
    if s in ["1", "2"]:
        return "1–2 Negative/Benign"
    if s == "3":
        return "3 Probably benign"
    if s in ["4", "4A", "4B", "4C"]:
        return "4 Suspicious"
    if s == "5":
        return "5 Highly suggestive"
    if s == "6":
        return "6 Known malignancy"
    return "Unknown"

df["BIRADS_main"] = df["BIRADSRisk"].apply(birads_main).astype("float")
df["BIRADS_coarse"] = df["BIRADSRisk"].apply(birads_coarse)

# Optional: binary variable for modeling (>=4 as "positive")
df["BIRADS_binary"] = np.where(df["BIRADS_main"] >= 4, "Positive (≥4)", "Negative (<4)")
df.loc[df["BIRADS_main"].isna(), "BIRADS_binary"] = "Unknown"

In [ ]:
# --- LesionType (coarse) ---

def lesiontype_coarse(x: str) -> str:
    if pd.isna(x):
        return "Other/Unknown"
    x = str(x).strip()

    # Explicit negative/benign wording
    if x in ["无异常", "无"] or "良性" in x or "增生" in x:
        return "Negative/Benign/None"

    if x in ["-", "—"]:
        return "Other/Unknown"

    has_mass   = ("肿块" in x) or ("肿块影" in x)
    has_nodule = ("结节" in x) or ("结节影" in x) or ("结节样" in x)
    has_calc   = ("钙化" in x) or ("钙化灶" in x)
    has_arch   = ("结构紊乱" in x) or ("结构扭曲" in x) or ("结构稍乱" in x)
    has_asym   = ("不对称" in x) or ("非对称" in x) or ("局灶性不对称" in x) or ("局灶不对称" in x)
    has_dense  = ("致密影" in x) or ("高密度影" in x)

    # Mixed if ≥2 major components
    major_flags = [has_mass or has_nodule, has_calc, has_arch or has_asym or has_dense]
    if sum(major_flags) >= 2:
        return "Mixed lesions"

    # Single-dominant patterns
    if has_mass or has_nodule:
        return "Mass/Nodule dominant"
    if has_calc:
        return "Calcification dominant"
    if has_arch or has_asym or has_dense:
        return "Architectural distortion/asymmetry"

    return "Other/Unknown"

df["LesionType_coarse"] = df["LesionType"].apply(lesiontype_coarse)

In [ ]:
# --- HistologicalSubtype (coarse) ---

malignant_keywords = ["癌", "Paget", "肉瘤"]   # covers 浸润性导管癌, 导管原位癌, 黏液癌, 癌肉瘤, etc.
benign_keywords = [
    "纤维腺瘤", "叶状肿瘤", "乳腺病", "腺病", "导管扩张",
    "炎", "囊肿", "囊性", "腺瘤", "硬化性腺病", "良性"
]

def histology_coarse(x: str) -> str:
    if pd.isna(x):
        return "Unknown/No surgery"
    s = str(x).strip()

    if any(k in s for k in ["无手术病理", "未手术", "我院未手术"]):
        return "Unknown/No surgery"

    if any(k in s for k in malignant_keywords):
        return "Malignant"

    if any(k in s for k in benign_keywords):
        return "Benign"

    # If it doesn't clearly fall into malignant/benign keywords, leave as unknown/other
    return "Unknown/Other"

df["Histology_coarse"] = df["HistologicalSubtype"].apply(histology_coarse)

In [ ]:
print("Density_coarse:")
print(df["Density_coarse"].value_counts(dropna=False), "\n")

print("BIRADS_coarse:")
print(df["BIRADS_coarse"].value_counts(dropna=False), "\n")

print("LesionType_coarse:")
print(df["LesionType_coarse"].value_counts(dropna=False), "\n")

print("Histology_coarse:")
print(df["Histology_coarse"].value_counts(dropna=False))

In [ ]:
meta_data_coarse = df.copy()

In [ ]:
# Let's try exclusion

meta_data_excluded = meta_data_coarse[
    meta_data_coarse['DensityCategory_std'].notna() &
    meta_data_coarse['BIRADSRisk_std'].notna()
]

print(meta_data_excluded['Group'].value_counts())

In [ ]:
# Check the file paths 
def find_all_dicoms(root_dir: str):
    """
    Recursively search for all DICOM files using pydicom.misc.is_dicom.
    Returns a list of verified DICOM file paths.
    """
    dicom_files = []
    for file in glob.glob(os.path.join(root_dir, "**"), recursive=True):
        if os.path.isfile(file) and is_dicom(file):
            dicom_files.append(file)
    return dicom_files

root_directory = "/data2/dh/MammographyData/SzOriginalFinal_2"
dicom_files = find_all_dicoms(root_directory)

print(f"✅ Found {len(dicom_files)} valid DICOM files.")
# for f in dicom_files[:5]:
#     print(f)

dicom_path_dict = {"TP": [], "TN": [], "FP": [], "FN": []}
for i in dicom_files:
    dicom_path_dict[i.split("/")[6]].append(i)

In [ ]:
print(len(dicom_path_dict['TP']))
print(len(dicom_path_dict['TN']))
print(len(dicom_path_dict['FP']))
print(len(dicom_path_dict['FN']))

In [ ]:
dicom_path_dict['FP']

In [ ]:
# Exclude 张美华_FN
for idx, row in meta_data_edited.iterrows():
    group = row['Group']
    patient_name = row['PatientName']
    dicom_list = dicom_path_dict.get(group, [])
    matched_files = [f for f in dicom_list if patient_name.split("_")[0].split()[0].split("（")[0] in f]
    if not matched_files:
        print(f"⚠️ No DICOM files found for Patient: {patient_name} in Group: {group}")

In [ ]:
meta_data_excluded['GroundTruth'] = [1 if row['Group'] in ['TP', 'FN'] else 0 for idx, row in meta_data_excluded.iterrows()]

In [ ]:
# meta_data_excluded.columns
print(meta_data_excluded['GroundTruth'].value_counts())
print(meta_data_excluded.columns.to_list())
meta_data_excluded.head()

In [ ]:
meta_data_sampled_TN = meta_data_excluded[meta_data_excluded['Group']=='TN'].sample(n=250, random_state=42)
meta_data_sampled_TP = meta_data_excluded[meta_data_excluded['Group']=='TP'].sample(n=250, random_state=42)
meta_data_sampled_FN = meta_data_excluded[meta_data_excluded['Group']=='FN'].sample(n=200, random_state=42)
meta_data_sampled_FP = meta_data_excluded[meta_data_excluded['Group']=='FP'].sample(n=200, random_state=42)

meta_data_sampled = pd.concat([meta_data_sampled_TN, meta_data_sampled_TP, 
                               meta_data_sampled_FN, meta_data_sampled_FP], ignore_index=True)

In [ ]:
print(meta_data_sampled['Group'].value_counts())

In [ ]:
df = meta_data_sampled.copy()

columns = ['PatientAge', 'DensityCategory_std', 'BIRADSRisk_std', 'LesionType_cat', 'HistologicalSubtype_cat', 
           'BIRADSRisk_sub', 'Density_coarse', 'BIRADS_main', 'BIRADS_coarse', 'BIRADS_binary', 'LesionType_coarse']
categorical = ['DensityCategory_std', 'BIRADSRisk_std', 'LesionType_cat', 'HistologicalSubtype_cat', 
               'BIRADSRisk_sub', 'Density_coarse', 'BIRADS_main', 'BIRADS_coarse', 'BIRADS_binary', 'LesionType_coarse']

# Reorder categories for each categorical variable by frequency
for col in ['LesionType_cat', 'LesionType_coarse', 'HistologicalSubtype_cat']:
    ordered_categories = df[col].value_counts(dropna=False).index
    df[col] = pd.Categorical(df[col], categories=ordered_categories, ordered=True)

nonnormal = None
mytable = TableOne(df, columns=columns, categorical=categorical, nonnormal=nonnormal, pval=False)

In [ ]:
mytable

In [ ]:
df = meta_data_sampled.copy()

columns = ['PatientAge', 'DensityCategory_std', 'BIRADSRisk_std', 'LesionType_cat', 'HistologicalSubtype_cat']
categorical = ['DensityCategory_std', 'BIRADSRisk_std', 'LesionType_cat', 'HistologicalSubtype_cat']

# Reorder categories for each categorical variable by frequency
for col in ['LesionType_cat', 'HistologicalSubtype_cat']:
    ordered_categories = df[col].value_counts(dropna=False).index
    df[col] = pd.Categorical(df[col], categories=ordered_categories, ordered=True)

nonnormal = None
mytable = TableOne(df, columns=columns, categorical=categorical, 
                   nonnormal=nonnormal, pval=True, groupby='Group')

In [ ]:
mytable

## DICOM paths 

In [ ]:
meta_data_sampled.head()

In [ ]:
meta_data_sampled_dicoms = pd.DataFrame()
meta_data_sampled_dicoms['anon_dicom_path'] = [
i 
for idx, row in meta_data_sampled.iterrows() 
for i in row['DICOM_paths']['DICOMPaths']
]
meta_data_sampled_dicoms['image_path'] = [i.replace("/data2/dh/MammographyData/SzOriginalFinal_2/SzOriginalCleaned", 
                                        "/hpc/home/ephdh/workspace/suzhou_false_validation/data/preprocessed_data").replace('.dcm', '.png') 
                                        for i in meta_data_sampled_dicoms['anon_dicom_path']]

In [ ]:
meta_data_sampled_dicoms.to_csv("/hpc/home/ephdh/workspace/suzhou_false_validation/data/meta_data/meta_data_sampled_dicoms.csv", 
                         index=False, encoding="utf-16")

In [ ]:
dicom_path_row_list = []

for idx, row in meta_data_sampled.iterrows():
    group = row['Group']
    patient_name = row['PatientName']
    dicom_list = dicom_path_dict.get(group, [])
    matched_files = [f for f in dicom_list if patient_name.split("_")[0].split()[0].split("（")[0] in f]
    
    if not matched_files:
        print(f"⚠️ No DICOM files found for Patient: {patient_name} in Group: {group}")
    else:
        if not len(matched_files) == 4:
            print(f"⚠️ Expected 4 DICOM files for Patient: {patient_name} in Group: {group}, but found {len(matched_files)}")

        dicom_path_row_list.append({
            "PatientName": patient_name,
            "Group": group,
            "DICOMPaths": matched_files
        })

meta_data_sampled['DICOM_paths'] = pd.Series(dicom_path_row_list)

In [ ]:
meta_data_sampled.to_csv("/hpc/home/ephdh/workspace/suzhou_false_validation/data/meta_data/meta_data_final_sampled.csv", 
                         index=False, encoding="utf-16")

In [ ]:
# import os

# def mirror_structure(source_dir, dest_dir):
#     """
#     Mirrors the directory structure from source_dir to dest_dir
#     without copying any files.
#     """
#     # specific Walk through the source directory
#     for dirpath, dirnames, filenames in os.walk(source_dir):
        
#         # 1. Create the relative path (e.g., "subfolder/nested")
#         rel_path = os.path.relpath(dirpath, source_dir)
        
#         # 2. Construct the full destination path
#         target_path = os.path.join(dest_dir, rel_path)
        
#         # 3. Create the directory (exist_ok=True prevents errors if it already exists)
#         os.makedirs(target_path, exist_ok=True)
        
#         print(f"Created: {target_path}")

# # --- Usage ---
# source = "/data2/dh/MammographyData/SzOriginalFinal_2/SzOriginalCleaned"  # Your source
# destination = "/hpc/home/ephdh/workspace/suzhou_false_validation/data/preprocessed_data" # Your target

# mirror_structure(source, destination)

## Scan dicoms for information

In [ ]:
meta_data_sampled_dicoms.head()

In [ ]:
def get_study_data(file_paths):
    """
    Scans a list of DICOM paths and returns a dictionary of study metadata.
    """
    data = {
        "institutions": set(),
        "dates": [],
        "vendors": set(),
        "models": set(),
        "study_uids": set(),
        "patient_ids": set()
    }

    print(f"Scanning {len(file_paths)} files...")

    for path in tqdm(file_paths, total=len(file_paths)):
        if not path or not os.path.exists(path):
            continue
            
        try:
            # Read header only
            ds = pydicom.dcmread(path, stop_before_pixels=True)
            
            # Extract tags safely
            if 'InstitutionName' in ds:
                data["institutions"].add(ds.InstitutionName)
                
            if 'StudyDate' in ds and ds.StudyDate:
                try:
                    data["dates"].append(datetime.strptime(ds.StudyDate, "%Y%m%d"))
                except ValueError:
                    pass 
            
            if 'Manufacturer' in ds:
                data["vendors"].add(ds.Manufacturer)
            
            if 'ManufacturerModelName' in ds:
                data["models"].add(ds.ManufacturerModelName)
                
            if 'StudyInstanceUID' in ds:
                data["study_uids"].add(ds.StudyInstanceUID)
                
            if 'PatientID' in ds:
                data["patient_ids"].add(ds.PatientID)

        except Exception:
            continue
            
    return data

In [ ]:
def generate_report(data):
    """
    Takes the dictionary from get_study_data and prints the formatted study details.
    """
    # 1. Process Dates
    if data['dates']:
        start_date = min(data['dates']).strftime('%d %B %Y')
        end_date = max(data['dates']).strftime('%d %B %Y')
    else:
        start_date, end_date = "Unknown", "Unknown"

    # 2. Format Lists
    inst_str = ", ".join(data['institutions']) if data['institutions'] else "[Insert Name of Hospital]"
    vendor_str = ", ".join(data['vendors']) if data['vendors'] else "[Insert Vendor]"
    model_str = ", ".join(data['models']) if data['models'] else "[Insert Model]"
    
    # 3. Print Report
    print("\n" + "="*60)
    print(" AUTOMATED STUDY INFORMATION REPORT")
    print("="*60)
    
    print("\n### Study Design and Ethical Considerations")
    print(f"This study was conducted at: {inst_str}")
    print(f"Data Collection Period:      {start_date} to {end_date}")
    
    print("\n### Study Population")
    print(f"Unique Studies (N):          {len(data['study_uids'])}")
    print(f"Unique Patients:             {len(data['patient_ids'])}")
    
    print("\n### Image Acquisition")
    print(f"Vendor:                      {vendor_str}")
    print(f"Model:                       {model_str}")
    
    print("\n### Missing Data (Fill Manually)")
    print("- IRB Reference Number")
    print("- Pathology/Biopsy Results (Ground Truth)")
    print("- Radiologist BI-RADS Scores")
    print("="*60 + "\n")

In [ ]:
dicom_paths = meta_data_sampled_dicoms['anon_dicom_path'].tolist()

In [ ]:
study_info = get_study_data(dicom_paths)

In [ ]:
generate_report(study_info)

In [ ]:
study_info